# 15b. Hybrid GAT + LightGBM, out-of-fold dạng ensemble (task 5.7, GĐ F)

Sửa lỗi của notebook 15: embedding train out-of-fold đến từ 5 GAT khác nhau nhưng val dùng embedding của một GAT thứ sáu (không gian lệch, F1 val 0,65 ở ToN). Ở đây val được dự đoán bằng ensemble: mỗi GAT nhóm cho một embedding val, LightGBM dự đoán từng góc rồi lấy trung bình xác suất. Ghi `model=hybrid_gat_lgbm_oofavg`, 3 seed × 2 bộ, chỉ `split=val`. **Không đọc test.**

Chẩn đoán trong `hybrid_search_oofavg_<bộ>.json`: F1 val của trung bình các góc, từng góc, và embedding của GAT huấn luyện trên toàn bộ train (cách cũ), với cùng GAT nhóm và LightGBM. Gắn output của 04, 11. **GPU T4**, bật Internet. Mô hình lưu ở `/kaggle/working/models_hybrid/` cho GĐ G, H.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])

## 1. pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Hybrid out-of-fold ensemble, mỗi bộ, 3 seed

In [ ]:
DATASETS = ["nf_bot_iot_v3", "nf_ton_iot_v3"]        # BoT trước (nhanh): lỗi sớm thì tốn ít giờ GPU
MODELS = f"{WORK}/models_hybrid"
for ds in DATASETS:
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    res, search = f"{WORK}/results_hybrid_oofavg_{ds}.csv", f"{WORK}/hybrid_search_oofavg_{ds}.json"
    for p in (res, search):
        assert not os.path.exists(p), f"{p} đã có từ trước"
    print("=====", ds, "| dữ liệu:", data_dir)
    with step(f"oofavg_{ds}"):
        run([sys.executable, "-I", "scripts/run_hybrid_oofavg.py", data_dir, ds, res, search, f"models={MODELS}"], cwd=REPO)

## 3. Kiểm tra và run_info.json (so sánh với baseline làm riêng ở notebook 15c từ file trong repo)

In [ ]:
import pandas as pd

crit, diag = {}, {}
for ds in DATASETS:
    res = pd.read_csv(f"{WORK}/results_hybrid_oofavg_{ds}.csv")
    cnt = res.groupby(["model", "seed"]).size()
    crit[ds] = {"du_3_seed_1_dong": bool(len(res) == 3 and (cnt == 1).all() and set(res["seed"]) == {42, 43, 44}),
                "chi_split_val": set(res["split"]) == {"val"},
                "f1_macro_mean": round(float(res["f1_macro"].mean()), 4), "fpr_mean": round(float(res["fpr"].mean()), 4),
                "size_kb_max": float(res["size_kb"].max()), "p95_ms_max": float(res["infer_ms_per_window"].max())}
    s = json.load(open(f"{WORK}/hybrid_search_oofavg_{ds}.json", encoding="utf-8"))["hybrid_gat_lgbm_oofavg"]["seeds"]
    diag[ds] = {k: v["diag"] for k, v in s.items()}
    print(ds, crit[ds], "\n", json.dumps(diag[ds], ensure_ascii=False, indent=1))
INFO["criteria"], INFO["diag"] = crit, diag
INFO["models_hybrid_dirs"] = sorted(os.listdir(MODELS))
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())